# Phase 4: Counterfactual Search Optimizer
In this notebook, we utilize the Cox Proportional Hazards model and the Latent Manifold Autoencoder to find biologically plausible counterfactual paths for high-risk patients. We use a PyTorch-native gradient descent optimizer operating in the latent space.


In [1]:
import torch
import pandas as pd
import numpy as np
import joblib
import os
import sys

sys.path.append('../src')
from optimizer import CounterfactualOptimizer
from data_pipeline import load_and_preprocess_data

# Load Data
print("Loading data...")
train_df, test_df, imm, dir_f, act = load_and_preprocess_data(filepath='../data/frmgham2.csv', random_seed=42)

# Load configuration
ae_config = joblib.load('../models/ae_config.joblib')
feature_cols = ae_config['feature_cols']

# Initialize Optimizer
opt = CounterfactualOptimizer(models_dir='../models')
scaler = opt.scaler
cont_cols = opt.cont_cols
cont_idx = opt.cont_idx
bin_cols = opt.bin_cols
bin_idx = opt.bin_idx


Loading data...
Loading data from ../data/frmgham2.csv...

Missingness before imputation (%):
PREVCHD       0.000000
educ          2.476415
HEARTRTE      0.023585
GLUCOSE       9.150943
TIMECHD       0.000000
time_years    0.000000
DIABETES      0.000000
PREVMI        0.000000
BPMEDS        1.250000
ANYCHD        0.000000
SYSBP         0.000000
SEX           0.000000
PREVHYP       0.000000
AGE           0.000000
TOTCHOL       1.179245
DIABP         0.000000
RANDID        0.000000
CIGPDAY       0.683962
BMI           0.448113
CURSMOKE      0.000000
PREVAP        0.000000
PREVSTRK      0.000000
dtype: float64

--- Imputation Impact (Train Set) ---
AGE: mean shift=0.00%, var shift=0.00%
TOTCHOL: mean shift=0.01%, var shift=1.14%
SYSBP: mean shift=0.00%, var shift=0.00%
DIABP: mean shift=0.00%, var shift=0.00%
CIGPDAY: mean shift=0.65%, var shift=0.28%
BMI: mean shift=0.01%, var shift=0.47%
HEARTRTE: mean shift=0.00%, var shift=0.03%
GLUCOSE: mean shift=0.41%, var shift=9.11%
SEX: mean shi

## 1. Selecting High-Risk Patients
We identify a subset of high-risk patients from the test cohort (e.g., those who experienced a cardiac event) to demonstrate the counterfactual search.


In [2]:
# Filter to those who had an event in the test set
test_events = test_df[test_df['ANYCHD'] == 1].copy()

# Pick 10 original patients as case studies (plus the 2 edge cases below makes 12)
sample_patients = test_events.head(10)

# Find a patient with very low SYSBP to test the hard bound
sysbp_scale_idx = cont_cols.index('SYSBP')
sysbp_thresh = (100 - scaler.mean_[sysbp_scale_idx]) / scaler.scale_[sysbp_scale_idx]
low_sysbp_patient = test_df[test_df['SYSBP'] < sysbp_thresh].head(1)

# Find a patient with low TOTCHOL to test the new monotonicity rule
totchol_scale_idx = cont_cols.index('TOTCHOL')
totchol_thresh = (140 - scaler.mean_[totchol_scale_idx]) / scaler.scale_[totchol_scale_idx]
low_totchol_patient = test_df[test_df['TOTCHOL'] < totchol_thresh].head(1)

# Combine them all
sample_patients = pd.concat([sample_patients, low_sysbp_patient, low_totchol_patient])
patient_vectors = sample_patients[feature_cols].values
print(f"Selected {len(patient_vectors)} high-risk patients for demonstration.")


Selected 12 high-risk patients for demonstration.


## 2. Running the Counterfactual Search
For each patient, the optimizer minimizes the Cox log partial hazard while enforcing proximity, immutability, and hard clinical bounds.


In [3]:
results = []

for i, p_vec in enumerate(patient_vectors):
    res = opt.optimize(p_vec)
    
    # We want to format the output nicely. Let's unscale the continuous features.
    x_orig_cont = res['x_orig'][cont_idx]
    x_cf_cont = res['x_cf'][cont_idx]
    
    # Inverse transform requires 2D array
    x_orig_native = scaler.inverse_transform(x_orig_cont.reshape(1, -1))[0]
    x_cf_native = scaler.inverse_transform(x_cf_cont.reshape(1, -1))[0]
    
    # Format changes
    changes = []
    
    # Continuous changes
    for feat, orig_val, cf_val in zip(cont_cols, x_orig_native, x_cf_native):
        diff = cf_val - orig_val
        if abs(diff) > 0.5: # Only show meaningful changes
            changes.append(f"{feat}: {orig_val:.1f} -> {cf_val:.1f} ({diff:+.1f})")
            
    # Binary changes
    x_orig_bin = res['x_orig'][bin_idx]
    x_cf_bin = np.round(res['x_cf'][bin_idx])
    for feat, orig_val, cf_val in zip(bin_cols, x_orig_bin, x_cf_bin):
        diff = cf_val - orig_val
        if diff != 0:
            changes.append(f"{feat}: {int(orig_val)} -> {int(cf_val)}")
            
    results.append({
        'Patient ID': f"Patient {i+1}",
        'Baseline Log Hazard': f"{res['baseline_hazard']:.3f}",
        'Post-Recourse Log Hazard': f"{res['final_hazard']:.3f}",
        'Hazard Drop': f"{res['hazard_drop']:.3f}",
        'Success': res['success'],
        'Suggested Interventions': " | ".join(changes) if (changes and res['success']) else "No feasible recourse found."
    })

pd.set_option('display.max_colwidth', None)
res_df = pd.DataFrame(results)
display(res_df)


,Patient ID,Baseline Log Hazard,Post-Recourse Log Hazard,Hazard Drop,Success,Suggested Interventions
0,Patient 1,0.208,0.151,0.057,True,TOTCHOL: 241.0 -> 237.2 (-3.8) | SYSBP: 126.5 -> 118.0 (-8.5) | DIABP: 78.5 -> 76.1 (-2.4) | CIGPDAY: 30.0 -> 28.9 (-1.1) | BMI: 29.9 -> 28.0 (-2.0) | HEARTRTE: 85.0 -> 82.4 (-2.6)
1,Patient 2,0.445,0.341,0.104,True,BMI: 28.9 -> 23.4 (-5.5) | HEARTRTE: 72.0 -> 61.2 (-10.8)
2,Patient 3,0.363,0.292,0.071,True,TOTCHOL: 272.0 -> 266.5 (-5.5) | SYSBP: 127.5 -> 121.2 (-6.3) | DIABP: 79.5 -> 75.2 (-4.3) | BMI: 26.4 -> 24.0 (-2.3) | HEARTRTE: 78.0 -> 73.9 (-4.1)
3,Patient 4,-0.019,-0.019,0.000,False,No feasible recourse found.
4,Patient 5,1.130,1.021,0.109,True,TOTCHOL: 231.0 -> 230.4 (-0.6) | SYSBP: 161.5 -> 156.4 (-5.1) | BMI: 31.0 -> 29.0 (-2.0) | HEARTRTE: 85.0 -> 83.7 (-1.3)
5,Patient 6,0.381,0.241,0.140,True,TOTCHOL: 305.0 -> 290.4 (-14.6) | DIABP: 75.0 -> 65.9 (-9.1) | HEARTRTE: 75.0 -> 70.9 (-4.1) | GLUCOSE: 98.0 -> 91.8 (-6.2)
6,Patient 7,-0.608,-0.826,0.217,True,TOTCHOL: 267.0 -> 265.8 (-1.2) | SYSBP: 119.0 -> 99.3 (-19.7) | BMI: 29.1 -> 25.8 (-3.4) | HEARTRTE: 83.0 -> 80.3 (-2.7) | GLUCOSE: 75.0 -> 70.8 (-4.2)
7,Patient 8,0.385,-0.223,0.608,True,SYSBP: 179.0 -> 163.6 (-15.4) | BMI: 44.5 -> 35.6 (-8.9) | GLUCOSE: 77.0 -> 65.0 (-12.0)
8,Patient 9,-0.024,-0.274,0.251,True,TOTCHOL: 315.0 -> 305.6 (-9.4) | SYSBP: 135.0 -> 127.7 (-7.3) | DIABP: 80.0 -> 71.2 (-8.8) | BMI: 25.2 -> 21.8 (-3.4) | HEARTRTE: 103.0 -> 100.7 (-2.3) | GLUCOSE: 89.0 -> 83.9 (-5.1) | CURSMOKE: 1 -> 0
9,Patient 10,0.353,0.317,0.037,False,No feasible recourse found.
